# Hands-on Task 1: Expense Calculator Agent

## Objective
Build a cyclic LangGraph agent that performs multi-step expense calculations using tools.

## Features
- Add expenses
- Apply tax
- Split total among people

## Workflow
Agent → Tool → Agent → Tool → Agent → END

## Tools
- add_expenses(a, b)
- apply_tax(amount, tax_percent)
- split_bill(amount, people)

In [1]:
import re
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

In [2]:
class ExpenseState(TypedDict, total=False):
    query: str
    a: float
    b: float
    tax_percent: float
    people: int
    total: float
    per_person: float
    added: bool
    taxed: bool
    split: bool
    next_tool: str
    final_answer: str
    messages: list

In [3]:
def add_expenses(a, b):
    return a + b


def apply_tax(amount, tax_percent):
    return amount + (amount * tax_percent / 100)


def split_bill(amount, people):
    return amount / people

In [4]:
def extract_expense_details(query):
    query_lower = query.lower()

    numbers = re.findall(r"\d+(?:\.\d+)?", query_lower)
    numbers = [float(num) for num in numbers]

    a = numbers[0] if len(numbers) > 0 else 0
    b = numbers[1] if len(numbers) > 1 else 0

    tax_match = re.search(r"(\d+(?:\.\d+)?)\s*%", query_lower)
    tax_percent = float(tax_match.group(1)) if tax_match else None

    people_match = re.search(r"(?:among|between|split among)\s+(\d+)", query_lower)
    people = int(people_match.group(1)) if people_match else None

    return a, b, tax_percent, people

In [5]:
def expense_agent(state: ExpenseState):

    query = state["query"]

    if "a" not in state or "b" not in state:
        a, b, tax_percent, people = extract_expense_details(query)

        return {
            "a": a,
            "b": b,
            "tax_percent": tax_percent,
            "people": people,
            "next_tool": "add_expenses",
            "messages": state.get("messages", []) + ["Agent decided to add expenses first."]
        }

    if not state.get("added", False):
        return {
            "next_tool": "add_expenses",
            "messages": state.get("messages", []) + ["Agent selected add_expenses tool."]
        }

    if state.get("tax_percent") is not None and not state.get("taxed", False):
        return {
            "next_tool": "apply_tax",
            "messages": state.get("messages", []) + ["Agent selected apply_tax tool."]
        }

    if state.get("people") is not None and not state.get("split", False):
        return {
            "next_tool": "split_bill",
            "messages": state.get("messages", []) + ["Agent selected split_bill tool."]
        }

    if state.get("split", False):
        answer = (
            f"Final amount after adding expenses and applying tax is ₹{state['total']:.2f}. "
            f"Each person should pay ₹{state['per_person']:.2f}."
        )
    else:
        answer = f"Final total amount is ₹{state['total']:.2f}."

    return {
        "next_tool": "done",
        "final_answer": answer,
        "messages": state.get("messages", []) + ["Agent completed the calculation."]
    }

In [6]:
def tool_executor(state: ExpenseState):

    tool = state["next_tool"]

    if tool == "add_expenses":
        total = add_expenses(state["a"], state["b"])

        return {
            "total": total,
            "added": True,
            "messages": state.get("messages", []) + [f"Tool add_expenses executed: {state['a']} + {state['b']} = {total}"]
        }

    elif tool == "apply_tax":
        total = apply_tax(state["total"], state["tax_percent"])

        return {
            "total": total,
            "taxed": True,
            "messages": state.get("messages", []) + [f"Tool apply_tax executed: total after tax = {total}"]
        }

    elif tool == "split_bill":
        per_person = split_bill(state["total"], state["people"])

        return {
            "per_person": per_person,
            "split": True,
            "messages": state.get("messages", []) + [f"Tool split_bill executed: each person pays {per_person}"]
        }

In [7]:
def route_expense_flow(state: ExpenseState):
    if state.get("next_tool") == "done":
        return END
    return "tool_executor"

In [8]:
builder = StateGraph(ExpenseState)

builder.add_node("expense_agent", expense_agent)
builder.add_node("tool_executor", tool_executor)

builder.add_edge(START, "expense_agent")

builder.add_conditional_edges(
    "expense_agent",
    route_expense_flow
)

builder.add_edge("tool_executor", "expense_agent")

expense_graph = builder.compile()

In [9]:
result = expense_graph.invoke(
    {
        "query": "What is the total of 1200 and 800?"
    }
)

print("Final Answer:")
print(result["final_answer"])

print("\nExecution Messages:")
for msg in result["messages"]:
    print("-", msg)

Final Answer:
Final total amount is ₹2000.00.

Execution Messages:
- Agent decided to add expenses first.
- Tool add_expenses executed: 1200.0 + 800.0 = 2000.0
- Agent completed the calculation.


In [10]:
result = expense_graph.invoke(
    {
        "query": "Add 2000 and 1500, apply 18% tax, and split among 3 people."
    }
)

print("Final Answer:")
print(result["final_answer"])

print("\nExecution Messages:")
for msg in result["messages"]:
    print("-", msg)

Final Answer:
Final amount after adding expenses and applying tax is ₹4130.00. Each person should pay ₹1376.67.

Execution Messages:
- Agent decided to add expenses first.
- Tool add_expenses executed: 2000.0 + 1500.0 = 3500.0
- Agent selected apply_tax tool.
- Tool apply_tax executed: total after tax = 4130.0
- Agent selected split_bill tool.
- Tool split_bill executed: each person pays 1376.6666666666667
- Agent completed the calculation.


# Hands-on Task 2: Travel Planner

## Objective
Build a memory-based chatbot that remembers user travel details across multiple turns.

## Details Remembered
- Destination
- Travel date
- Preference

## Concepts Used
- MemorySaver
- thread_id
- Context retention

In [11]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

try:
    from langgraph.checkpoint.memory import MemorySaver
except ImportError:
    from langgraph.checkpoint.memory import InMemorySaver as MemorySaver

In [12]:
class TravelState(TypedDict, total=False):
    user_input: str
    destination: str
    preference: str
    travel_date: str
    response: str

In [13]:
def travel_memory_node(state: TravelState):

    user_input = state["user_input"].lower()

    updates = {}

    if "bali" in user_input:
        updates["destination"] = "Bali"

    if "budget" in user_input:
        updates["preference"] = "budget hotels"

    elif "luxury" in user_input:
        updates["preference"] = "luxury hotels"

    if "december" in user_input:
        updates["travel_date"] = "December"

    elif "january" in user_input:
        updates["travel_date"] = "January"

    if "summarise" in user_input or "summarize" in user_input:
        destination = state.get("destination", "the selected destination")
        travel_date = state.get("travel_date", "the selected month")
        preference = state.get("preference", "your preferred stay option")

        updates["response"] = (
            f"You are planning a trip to {destination} in {travel_date} "
            f"and prefer {preference}."
        )

    else:
        updates["response"] = "Got it. I have updated your travel plan."

    return updates

In [14]:
memory = MemorySaver()

builder = StateGraph(TravelState)

builder.add_node("travel_memory_node", travel_memory_node)

builder.add_edge(START, "travel_memory_node")
builder.add_edge("travel_memory_node", END)

travel_graph = builder.compile(checkpointer=memory)

In [15]:
config = {
    "configurable": {
        "thread_id": "travel_user_1"
    }
}

In [16]:
result = travel_graph.invoke(
    {
        "user_input": "I am planning a trip to Bali."
    },
    config=config
)

print(result["response"])

Got it. I have updated your travel plan.


In [17]:
result = travel_graph.invoke(
    {
        "user_input": "I prefer budget hotels."
    },
    config=config
)

print(result["response"])

Got it. I have updated your travel plan.


In [18]:
result = travel_graph.invoke(
    {
        "user_input": "My trip is in December."
    },
    config=config
)

print(result["response"])

Got it. I have updated your travel plan.


In [19]:
result = travel_graph.invoke(
    {
        "user_input": "Summarise my travel plan."
    },
    config=config
)

print(result["response"])

You are planning a trip to Bali in December and prefer budget hotels.


# Hands-on Task 3: Access Control Approval System

## Objective
Build a Human-in-the-Loop access approval workflow.

## Features
- Grant access
- Revoke access
- Pause for manager approval before execution

## Concepts Used
- interrupt()
- Human approval
- Tool execution after approval


In [20]:
import re
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command

try:
    from langgraph.checkpoint.memory import MemorySaver
except ImportError:
    from langgraph.checkpoint.memory import InMemorySaver as MemorySaver

In [21]:
def grant_access(user, system):
    return f"Access granted: {user} has been given access to {system}."


def revoke_access(user, system):
    return f"Access revoked: {user}'s access to {system} has been removed."

In [22]:
class AccessState(TypedDict, total=False):
    request: str
    user: str
    system: str
    action: str
    approved: bool
    result: str
    response: str

In [23]:
def parse_access_request(state: AccessState):

    request = state["request"]

    request_lower = request.lower()

    if "revoke" in request_lower or "remove" in request_lower:
        action = "revoke"
    else:
        action = "grant"

    user_match = re.search(r"user\s+'([^']+)'", request)

    if user_match:
        user = user_match.group(1)
    else:
        user = "Unknown User"

    if "database admin" in request_lower:
        system = "database admin"
    elif "database" in request_lower:
        system = "database"
    else:
        system = "requested system"

    return {
        "user": user,
        "system": system,
        "action": action,
        "response": f"Prepared request to {action} {system} access for {user}."
    }

In [24]:
def approval_node(state: AccessState):

    approval = interrupt(
        f"Approve request to {state['action']} {state['system']} access for {state['user']}? Type yes or no."
    )

    approved = str(approval).lower().strip() == "yes"

    return {
        "approved": approved
    }

In [25]:
def approval_router(state: AccessState):

    if state.get("approved") is True:
        return "access_tool_node"

    return "rejected_node"

In [26]:
def access_tool_node(state: AccessState):

    if state["action"] == "grant":
        result = grant_access(state["user"], state["system"])

    else:
        result = revoke_access(state["user"], state["system"])

    return {
        "result": result,
        "response": result
    }

In [27]:
def rejected_node(state: AccessState):

    return {
        "result": "Request rejected by manager.",
        "response": "Access request was rejected. No action was executed."
    }

In [28]:
access_memory = MemorySaver()

builder = StateGraph(AccessState)

builder.add_node("parse_access_request", parse_access_request)
builder.add_node("approval_node", approval_node)
builder.add_node("access_tool_node", access_tool_node)
builder.add_node("rejected_node", rejected_node)

builder.add_edge(START, "parse_access_request")
builder.add_edge("parse_access_request", "approval_node")

builder.add_conditional_edges(
    "approval_node",
    approval_router
)

builder.add_edge("access_tool_node", END)
builder.add_edge("rejected_node", END)

access_graph = builder.compile(checkpointer=access_memory)

In [29]:
access_memory = MemorySaver()

builder = StateGraph(AccessState)

builder.add_node("parse_access_request", parse_access_request)
builder.add_node("approval_node", approval_node)
builder.add_node("access_tool_node", access_tool_node)
builder.add_node("rejected_node", rejected_node)

builder.add_edge(START, "parse_access_request")
builder.add_edge("parse_access_request", "approval_node")

builder.add_conditional_edges(
    "approval_node",
    approval_router
)

builder.add_edge("access_tool_node", END)
builder.add_edge("rejected_node", END)

access_graph = builder.compile(checkpointer=access_memory)

In [31]:
access_config = {
    "configurable": {
        "thread_id": "access_request_1"
    }
}

In [32]:
result = access_graph.invoke(
    {
        "request": "Give database admin access to user 'Kiran'"
    },
    config=access_config
)

print(result)

{'request': "Give database admin access to user 'Kiran'", 'user': 'Kiran', 'system': 'database admin', 'action': 'grant', 'response': 'Prepared request to grant database admin access for Kiran.', '__interrupt__': [Interrupt(value='Approve request to grant database admin access for Kiran? Type yes or no.', id='582f7e4bf6bf90575685179644f69339')]}


In [33]:
result = access_graph.invoke(
    Command(resume="yes"),
    config=access_config
)

print(result["response"])

Access granted: Kiran has been given access to database admin.
